# Boston Housing Price Prediction - Regression with MLflow & DagsHub

## 1. Importing Packages

In [4]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

try:
    from xgboost import XGBRegressor
    has_xgboost = True
except ImportError:
    has_xgboost = False

import mlflow
import mlflow.sklearn
if has_xgboost:
    import mlflow.xgboost
import dagshub
import os

## 2. DagsHub MLflow Setup

In [5]:


dagshub.init(
    repo_owner="ssudharsan0609",
    repo_name="Lab3DevOps",
    mlflow=True
)

mlflow.set_experiment("Boston Housing Price Regression")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=66587e76-81d9-49cb-9c5c-23453c669dbd&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=3ed50f817fa7e9ef43154d58012d4fe34aa9f4553fdd14d7c7e8e1f4a973f280




Accessing as ssudharsan0609

Initialized MLflow to track repo "ssudharsan0609/Lab3DevOps"

Repository ssudharsan0609/Lab3DevOps initialized!

2026/08/07 08:30:30 INFO mlflow.tracking.fluent: Experiment with name 'Boston Housing Price Regression' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/784e089c79aa421d9645168310aa9873', creation_time=1786091430809, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1786091430809, lifecycle_stage='active', name='Boston Housing Price Regression', tags={}, trace_location=None, workspace='default'>

## 3. Data Loading and Preprocessing

In [6]:
# Load Boston Housing Dataset
url = "https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv"
df = pd.read_csv(url)

print("Dataset Shape:", df.shape)
print("Features:", list(df.columns))
df.head()

Dataset Shape: (506, 14)
Features: ['crim', 'zn', 'indus', 'chas', 'nox', 'rm', 'age', 'dis', 'rad', 'tax', 'ptratio', 'b', 'lstat', 'medv']


,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,b,lstat,medv
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2


In [7]:
X = df.drop(columns=["medv"])
y = df["medv"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train_scaled.shape)
print("Test shape:", X_test_scaled.shape)

Train shape: (404, 13)
Test shape: (102, 13)


## 4. Define 5 Regression Models

In [8]:
model_5 = (
    "XGBoost Regressor",
    XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42),
    X_train_scaled,
    X_test_scaled
) if has_xgboost else (
    "AdaBoost Regressor",
    AdaBoostRegressor(n_estimators=100, random_state=42),
    X_train_scaled,
    X_test_scaled
)

models = [
    (
        "Linear Regression",
        LinearRegression(),
        X_train_scaled,
        X_test_scaled
    ),
    (
        "Ridge Regression",
        Ridge(alpha=1.0, random_state=42),
        X_train_scaled,
        X_test_scaled
    ),
    (
        "Random Forest Regressor",
        RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42),
        X_train_scaled,
        X_test_scaled
    ),
    (
        "Gradient Boosting Regressor",
        GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
        X_train_scaled,
        X_test_scaled
    ),
    model_5
]

## 5. Train Models & Evaluate Regression Metrics

In [9]:
def eval_metrics(y_true, y_pred, n_features):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    n = len(y_true)
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - n_features - 1)
    return {
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "Adjusted_R2": adj_r2
    }

results = []
trained_models = []

for name, model, X_tr, X_te in models:
    model.fit(X_tr, y_train)
    preds = model.predict(X_te)
    metrics = eval_metrics(y_test, preds, X_tr.shape[1])
    
    results.append(metrics)
    trained_models.append(model)
    
    print("=" * 50)
    print(f"Model: {name}")
    print("=" * 50)
    for m_name, m_val in metrics.items():
        print(f"{m_name:15s}: {m_val:.4f}")
    print()

Model: Linear Regression
MSE            : 24.2911
RMSE           : 4.9286
MAE            : 3.1891
R2             : 0.6688
Adjusted_R2    : 0.6198

Model: Ridge Regression
MSE            : 24.3129
RMSE           : 4.9308
MAE            : 3.1857
R2             : 0.6685
Adjusted_R2    : 0.6195

Model: Random Forest Regressor
MSE            : 8.9223
RMSE           : 2.9870
MAE            : 2.2160
R2             : 0.8783
Adjusted_R2    : 0.8604

Model: Gradient Boosting Regressor
MSE            : 6.2082
RMSE           : 2.4916
MAE            : 1.9122
R2             : 0.9153
Adjusted_R2    : 0.9028

Model: XGBoost Regressor
MSE            : 6.1215
RMSE           : 2.4742
MAE            : 1.8468
R2             : 0.9165
Adjusted_R2    : 0.9042



## 6. Log All Experiments to DagsHub via MLflow

In [10]:
for i, (name, model, _, _) in enumerate(models):
    metrics = results[i]
    
    with mlflow.start_run(run_name=name):
        mlflow.log_param("Model", name)
        if hasattr(model, "get_params"):
            mlflow.log_params(model.get_params())
            
        for m_name, m_val in metrics.items():
            mlflow.log_metric(m_name, float(m_val))
            
        if "XGBoost" in name and has_xgboost:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")

print("All 5 regression experiments logged successfully to DagsHub!")

2026/08/07 08:30:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Linear Regression at: https://dagshub.com/ssudharsan0609/Lab3DevOps.mlflow/#/experiments/0/runs/f3548870bf194945bcc58b921936a10a
🧪 View experiment at: https://dagshub.com/ssudharsan0609/Lab3DevOps.mlflow/#/experiments/0


2026/08/07 08:31:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Ridge Regression at: https://dagshub.com/ssudharsan0609/Lab3DevOps.mlflow/#/experiments/0/runs/860be93bf9664ebb808f264fda907978
🧪 View experiment at: https://dagshub.com/ssudharsan0609/Lab3DevOps.mlflow/#/experiments/0


2026/08/07 08:32:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest Regressor at: https://dagshub.com/ssudharsan0609/Lab3DevOps.mlflow/#/experiments/0/runs/a82a260db58d4e21a03f0966ef0806e8
🧪 View experiment at: https://dagshub.com/ssudharsan0609/Lab3DevOps.mlflow/#/experiments/0


2026/08/07 08:33:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Gradient Boosting Regressor at: https://dagshub.com/ssudharsan0609/Lab3DevOps.mlflow/#/experiments/0/runs/50aa32a21b8c428bbdcd70b0d280031f
🧪 View experiment at: https://dagshub.com/ssudharsan0609/Lab3DevOps.mlflow/#/experiments/0


2026/08/07 08:34:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost Regressor at: https://dagshub.com/ssudharsan0609/Lab3DevOps.mlflow/#/experiments/0/runs/f6ef192a5049477caae2b1f808c32e3e
🧪 View experiment at: https://dagshub.com/ssudharsan0609/Lab3DevOps.mlflow/#/experiments/0
All 5 regression experiments logged successfully to DagsHub!


## 7. Identify Champion Model & Register to DagsHub Model Registry

In [12]:
best_idx = np.argmax([r["R2"] for r in results])
best_model_name = models[best_idx][0]
best_model = trained_models[best_idx]
best_metrics = results[best_idx]

print(f"Champion Model Selected: {best_model_name}")
print(f"Best R2 Score: {best_metrics['R2']:.4f}")

with mlflow.start_run(run_name=f"Champion_{best_model_name}") as run:
    mlflow.log_param("Model", best_model_name)
    
    for m_name, m_val in best_metrics.items():
        mlflow.log_metric(m_name, float(m_val))
        
    if "XGBoost" in best_model_name and has_xgboost:
        mlflow.xgboost.log_model(
            best_model,
            "model",
            registered_model_name="Boston_Housing_Champion_Model"
        )
    else:
        mlflow.sklearn.log_model(
            best_model,
            "model",
            registered_model_name="Boston_Housing_Champion_Model"
        )
        
    run_id = run.info.run_id

print(f"Successfully registered Champion Model '{best_model_name}' under 'Boston_Housing_Champion_Model'!")
print(f"Run ID: {run_id}")

Champion Model Selected: XGBoost Regressor
Best R2 Score: 0.9165


2026/08/07 08:38:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'Boston_Housing_Champion_Model'.
2026/08/07 08:38:45 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Boston_Housing_Champion_Model, version 1
Created version '1' of model 'Boston_Housing_Champion_Model'.


🏃 View run Champion_XGBoost Regressor at: https://dagshub.com/ssudharsan0609/Lab3DevOps.mlflow/#/experiments/0/runs/65df14c728b045a7a394a80e0dbb651f
🧪 View experiment at: https://dagshub.com/ssudharsan0609/Lab3DevOps.mlflow/#/experiments/0
Successfully registered Champion Model 'XGBoost Regressor' under 'Boston_Housing_Champion_Model'!
Run ID: 65df14c728b045a7a394a80e0dbb651f
